In [ ]:
# ============================================================
# CELL 1: CÀI ĐẶT THƯ VIỆN
# ============================================================
!pip install -q transformers==4.46.3 peft==0.10.0 accelerate sentencepiece
!pip install -q fastapi uvicorn nest-asyncio pyngrok
!pip install -q sentence-transformers underthesea scikit-learn
print('✅ Đã cài đặt xong tất cả thư viện!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 109.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 90.5 MB/s eta 0:00:00
✅ Đã cài đặt xong tất cả thư viện!


In [ ]:
# ============================================================
# CELL 2: KẾT NỐI GOOGLE DRIVE
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive đã được kết nối!')

Mounted at /content/drive
✅ Drive đã được kết nối!


In [ ]:
# ============================================================
# CELL 3: CẤU HÌNH ĐƯỜNG DẪN & MODEL
# ============================================================
import os

BASE_DRIVE_PATH = '/content/drive/MyDrive/Final-Statistical-Learning'
SUMMARIZATION_ADAPTER_PATH = f'{BASE_DRIVE_PATH}/Summarization/vit5-summarization-adapter'
BASE_MODEL_NAME = 'VietAI/vit5-large'

# Cấu hình model cho Hybrid Search & Reranking
EMBEDDING_MODEL_NAME = 'BAAI/bge-m3'
RERANKER_MODEL_NAME = 'BAAI/bge-reranker-v2-m3'

print('📁 Kiểm tra đường dẫn Adapter Summarization:')
print(f'  {"✅" if os.path.exists(SUMMARIZATION_ADAPTER_PATH) else "❌ KHÔNG TỒN TẠI"} - {SUMMARIZATION_ADAPTER_PATH}')

📁 Kiểm tra đường dẫn Adapter Summarization:
  ✅ - /content/drive/MyDrive/Final-Statistical-Learning/Summarization/vit5-summarization-adapter


In [ ]:
# ============================================================
# CELL 4: LOAD CÁC MODEL (SUMMARIZATION + EMBEDDING + RERANKER)
# ============================================================
import torch
from transformers import T5Tokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel
from sentence_transformers import SentenceTransformer, CrossEncoder

# 1. Load Summarization Model
print('🔄 Đang load Tokenizer (ViT5)...')
tokenizer = T5Tokenizer.from_pretrained(BASE_MODEL_NAME, legacy=False)

print('🔄 Đang load Summarization Base Model...')
summ_base_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto'
)
print('🔄 Đang gắn Summarization LoRA Adapter...')
summ_model = PeftModel.from_pretrained(summ_base_model, SUMMARIZATION_ADAPTER_PATH)
summ_model.eval()
print('✅ Summarization Model sẵn sàng!\n')

# 2. Load Embedding Model (Bi-Encoder)
print(f'🔄 Đang load Embedding Model: {EMBEDDING_MODEL_NAME}...')
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print('✅ Embedding Model sẵn sàng!\n')

# 3. Load Reranker Model (Cross-Encoder)
print(f'🔄 Đang load Cross-Encoder Model: {RERANKER_MODEL_NAME}...')
reranker_model = CrossEncoder(RERANKER_MODEL_NAME)
print('✅ Reranker Model sẵn sàng!\n\n')

print('🎉 TẤT CẢ MODEL ĐÃ SẴN SÀNG!')

🔄 Đang load Tokenizer (ViT5)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


spiece.model:   0%|          | 0.00/820k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/640 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

✅ Reranker Model sẵn sàng!


🎉 TẤT CẢ MODEL ĐÃ SẴN SÀNG!


In [ ]:
# ============================================================
# CELL 5: HÀM INFERENCE - SUMMARIZATION
# ============================================================

def summarize_text(text: str, max_length: int = 200) -> str:
    input_text = 'tóm tắt: ' + text

    inputs = tokenizer(
        input_text,
        return_tensors='pt',
        max_length=1024,
        truncation=True
    ).to(summ_model.device)

    with torch.no_grad():
        outputs = summ_model.generate(
            **inputs,
            max_length=max_length,
            # min_length=30,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=2
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# ============================================================
# CELL 6: DENSE RETRIEVAL & CROSS-ENCODER RERANKING
# ============================================================
import numpy as np
from typing import List

def select_relevant_chunks(
    chunks: List[str],
    chunk_headings: List[str],
    question: str,
    top_k: int = 3,
    candidate_pool_size: int = 15
) -> List[str]:
    """
    Quy trình: Contextual Chunking -> Bi-Encoder (Top 15) -> Cross-Encoder (Top K).
    """

    if not chunks:
        return []

    # Giới hạn top_k không vượt quá số lượng chunk hiện có
    actual_top_k = min(top_k, len(chunks))
    if len(chunks) <= actual_top_k:
        return chunks

    # 1. Contextual Chunking: Ghép Heading vào Chunk để tăng độ hiểu ngữ cảnh cho Model
    enhanced_chunks = []
    for i, chunk in enumerate(chunks):
        heading = chunk_headings[i] if chunk_headings and i < len(chunk_headings) else ""
        if heading:
            enhanced_chunks.append(f"Tiêu đề: {heading}\nNội dung: {chunk}")
        else:
            enhanced_chunks.append(chunk)

    # 2. Stage 1: Retrieval bằng Bi-Encoder (Dense Search)
    query_emb = embedding_model.encode(question, normalize_embeddings=True)
    chunk_embs = embedding_model.encode(enhanced_chunks, normalize_embeddings=True, batch_size=32, show_progress_bar=False)

    # Tính Cosine Similarity bằng phép nhân ma trận (cực nhanh nhờ numpy)
    sims = np.dot(chunk_embs, query_emb)

    # Lấy ra index của top các ứng viên (ví dụ: top 15)
    actual_pool_size = min(candidate_pool_size, len(chunks))
    top_candidate_indices = np.argsort(sims)[::-1][:actual_pool_size]

    candidate_chunks = [enhanced_chunks[i] for i in top_candidate_indices]

    # 3. Stage 2: Reranking bằng Cross-Encoder
    # Bắt cặp câu hỏi với từng chunk ứng viên để model "đọc kỹ" lại
    pairs = [[question, cand] for cand in candidate_chunks]
    ce_scores = reranker_model.predict(pairs)

    # Lấy ra index của top_k có điểm cao nhất từ Cross-Encoder
    reranked_indices = np.argsort(ce_scores)[::-1][:actual_top_k]

    # 4. Finalize
    # Map ngược lại index gốc của tài liệu
    final_original_indices = [top_candidate_indices[i] for i in reranked_indices]

    # Sắp xếp lại theo thứ tự index gốc để ngữ cảnh liền mạch khi đưa vào LLM
    final_original_indices.sort()

    # Trả về các chunk gốc (không kèm chữ "Tiêu đề:" để LLM đọc tự nhiên hơn)
    return [chunks[i] for i in final_original_indices]

print("✅ Logic Dense Retrieval & Reranking (AI-based) đã sẵn sàng!")

✅ Logic Dense Retrieval & Reranking (AI-based) đã sẵn sàng!


In [ ]:
# ============================================================
# CELL 7: ĐỊNH NGHĨA API FASTAPI
# ============================================================
import threading
import time
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional, List

app = FastAPI(title='RAG Serving API', version='2.1.0')

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

# --- Schemas ---
class SummarizeRequest(BaseModel):
    text: str
    max_length: Optional[int] = 200

class SummarizeResponse(BaseModel):
    summary: str
    chunk_count: int

# Schema mới cập nhật theo chuẩn BE của bạn
class RetrieveRequest(BaseModel):
    question: str
    chunks: List[str]
    chunk_headings: Optional[List[str]] = []
    top_k: Optional[int] = 3

class RetrieveResponse(BaseModel):
    relevant_chunks: List[str]

# --- Endpoints ---
@app.get('/')
def health_check():
    return {'status': 'ok', 'models': ['summarization', 'bi-encoder', 'cross-encoder']}

@app.post('/summarize', response_model=SummarizeResponse)
def api_summarize(req: SummarizeRequest):
    if not req.text or not req.text.strip():
        raise HTTPException(status_code=400, detail='Văn bản rỗng.')
    try:
        chunks = [c.strip() for c in req.text.split('|||') if c.strip()]
        if not chunks: chunks = [req.text]

        summaries = [summarize_text(chunk, req.max_length) for chunk in chunks]
        final_text = ' '.join(summaries)

        if len(chunks) > 1 and len(final_text) > 500:
            final_summary = summarize_text(final_text, req.max_length)
        else:
            final_summary = final_text

        return SummarizeResponse(summary=final_summary, chunk_count=len(chunks))
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post('/retrieve', response_model=RetrieveResponse)
def api_retrieve(req: RetrieveRequest):
    if not req.question.strip() or not req.chunks:
        raise HTTPException(status_code=400, detail='Câu hỏi và mảng chunks không được rỗng.')
    try:
        # Gọi hàm xử lý AI
        results = select_relevant_chunks(
            chunks=req.chunks,
            chunk_headings=req.chunk_headings,
            question=req.question,
            top_k=req.top_k
        )
        return RetrieveResponse(relevant_chunks=results)
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Lỗi khi retrieve: {str(e)}")

In [ ]:
# ============================================================
# CELL 8: KHỞI ĐỘNG SERVER
# ============================================================
PORT = 8001

def start_fastapi_server():
    config = uvicorn.Config(app, host='0.0.0.0', port=PORT, log_level='warning')
    server = uvicorn.Server(config)
    server.run()

server_thread = threading.Thread(target=start_fastapi_server, daemon=True)
server_thread.start()

time.sleep(5)
print(f'✅ [SUCCESS] FastAPI Server đã kích hoạt tại cổng {PORT}!')

✅ [SUCCESS] FastAPI Server đã kích hoạt tại cổng 8001!


In [ ]:
# ============================================================
# CELL 9: EXPOSE API QUA TUNNEL (PINGGY)
# ============================================================
# Chạy lệnh sau trên terminal của colab
# ssh -p 443 -R 0:localhost:8001 a.pinggy.io